# AstroRAG — Embeddings + FAISS

Neste notebook iremos:

- gerar embeddings dos chunks;
- criar banco vetorial FAISS;
- implementar busca semântica;
- preparar o pipeline RAG.

In [1]:
# Execute apenas se necessário

!pip install sentence-transformers
!pip install faiss-cpu
!pip install numpy

# Importações

In [2]:
import pickle
import numpy as np
import faiss

from sentence_transformers import SentenceTransformer

# Carregando chunks

In [3]:
with open("chunks.pkl", "rb") as f:
    chunks = pickle.load(f)

print(f"Quantidade de chunks carregados: {len(chunks)}")

Quantidade de chunks carregados: 1763


# Carregando modelo de embeddings

In [4]:
model = SentenceTransformer(
    'sentence-transformers/all-MiniLM-L6-v2'
)

print("Modelo carregado com sucesso!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Modelo carregado com sucesso!


# Extraindo textos dos chunks

In [5]:
texts = [chunk.page_content for chunk in chunks]

print(texts[0][:500])

Draft version December 24, 2021
Typeset using LATEX default style in AASTeX63
A Machine Learning-based Direction-of-origin Filter for the Identiﬁcation of Radio Frequency
Interference in the Search for Technosignatures
Pavlo Pinchuk1 and Jean-Luc Margot 2, 1
1Department of Physics and Astronomy, University of California, Los Angeles, CA 90095, USA
2Department of Earth, Planetary, and Space Sciences, University of California, Los Angeles, CA 90095, USA


# Gerando embeddings

In [6]:
embeddings = model.encode(
    texts,
    show_progress_bar=True
)

print(embeddings.shape)

Batches:   0%|          | 0/56 [00:00<?, ?it/s]

(1763, 384)


# Criando índice FAISS

In [7]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(np.array(embeddings))

print(f"Quantidade de vetores no índice: {index.ntotal}")

Quantidade de vetores no índice: 1763


# Função de busca semântica

In [8]:
def semantic_search(query, k=3):

    query_embedding = model.encode([query])

    distances, indices = index.search(
        np.array(query_embedding),
        k
    )

    results = []

    for idx in indices[0]:
        results.append(texts[idx])

    return results

# Teste de busca

In [9]:
query = "What are Fast Radio Bursts?"

results = semantic_search(query)

for i, result in enumerate(results):

    print(f"\nRESULTADO {i+1}")
    print("="*80)
    print(result[:1500])


RESULTADO 1
Astron. Astrophys. Rev., 30(1):2, 2022.
[2] L. G. Spitler et al. A Repeating Fast Radio Burst.Nature, 531:202, 2016.
[3] Bridget C. Andersen et al. CHIME/FRB Discovery of 25 Repeating Fast Radio Burst Sources.
Astrophys. J., 947(2):83, 2023.
[4] Adam E. Lanman et al. A Sudden Period of High Activity from Repeating Fast Radio Burst
20201124A.Astrophys. J., 927(1):59, 2022.
[5] F. Kirsten et al. A link between repeating and non-repeating fast radio bursts through their

RESULTADO 2
THE PHYSICS OF FAST RADIO BURSTS
Bing Zhang
Nevada Center for Astrophysics and Department of Physics and Astronomy,
University of Nevada Las Vegas,
Nevada 89154 USA
(Dated: published 25 September 2023)
Fast radio bursts (FRBs), millisecond-duration bursts prevailing in the radio sky, are
the latest big puzzle in the universe and have been a subject of intense observational
and theoretical investigations in recent years. The rapid accumulation of the observa-

RESULTADO 3
N. Wang, J. Yang, and J. P